# FairAgent — Phase 4: Selection-Trial Infrastructure + Pilot

Run this notebook on a Kaggle **GPU** session (T4 x1 is enough for Llama-3.2-3B-Instruct).

Before running Cell 2, attach (via "Add Input" in the right sidebar):
- Your private Kaggle Dataset holding the pre-pulled Llama-3.2-3B-Instruct weights (Phase 0), **or** set `MODEL_PATH` to the HF Hub id and add your HF token as a Kaggle Secret.
- Your `results/` Kaggle Dataset, if this isn't the first Phase 4 session (Cell 3 restores prior progress from it).

Every cell is idempotent / resumable — re-running Cell 4 continues where the last session left off (session-discipline pattern, see the Implementation Plan).

## Cell 1 — Fresh clone + install deps (bootstrap, under 2 min)

In [ ]:
import os, shutil, sys

REPO_DIR = "/kaggle/working/fairagent"

# Always leave REPO_DIR BEFORE deleting it. If cwd is left pointing inside
# a directory that then gets rmtree'd, os.getcwd() breaks permanently for
# this kernel process, and every subsequent `!shell` command fails at
# shell-init (getcwd) before it even runs — the "cannot access parent
# directories" error. Re-running this cell in an already-running kernel
# (cwd == REPO_DIR from a prior run) is exactly when that happens.
os.chdir("/kaggle/working")

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)  # always force-fresh clone, never reuse a stale dir

BRANCH = "claude/fairagent-llm-bias-audit-24vjtj"  # switch to "main" once merged

!git clone https://github.com/Madan-Sai-9/fairagent.git {REPO_DIR}
!cd {REPO_DIR} && git checkout {BRANCH}
!pip install -q transformers accelerate pyyaml pandas statsmodels scipy

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


## Cell 2 — Load Llama-3.2-3B-Instruct

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Point at your attached Kaggle Dataset's folder (Phase 0 pre-pulled weights),
# or an HF Hub id if you'd rather download fresh (needs an HF token Secret
# if the repo is gated).
MODEL_PATH = "/kaggle/input/<your-llama-3.2-3b-dataset-slug>"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print("loaded on", device)


## Cell 3 — Restore prior progress (skip on your very first Phase 4 session)

Copies last session's result CSVs from your attached `results/` Kaggle Dataset into
the fresh clone's `results/` folder, so `run_pilot()` sees what's already done and
skips those `trial_id`s.

In [ ]:
import os, shutil

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)  # defensive: re-assert cwd in case the kernel restarted
os.makedirs("results", exist_ok=True)

PRIOR_RESULTS_DATASET = "/kaggle/input/<your-results-dataset-slug>"

if os.path.isdir(PRIOR_RESULTS_DATASET):
    restored_any = False
    for fname in ("phase4_pilot_trials.csv", "phase4_pilot_clients.csv"):
        src = os.path.join(PRIOR_RESULTS_DATASET, fname)
        if os.path.exists(src):
            shutil.copy(src, os.path.join("results", fname))
            print("restored", fname)
            restored_any = True
    if not restored_any:
        print(f"{PRIOR_RESULTS_DATASET} is attached but has no phase4_pilot_*.csv in it yet — starting fresh")
else:
    print(f"'{PRIOR_RESULTS_DATASET}' not found (dataset not attached, or you haven't "
          f"created it yet) — starting fresh. This is expected on your first Phase 4 session.")

print("cwd:", os.getcwd(), "| results/ contents:", os.listdir("results"))


## Cell 4 — Run one bounded batch

Stops at 60 new trials or 70 minutes wall-clock (`configs/phase4_pilot.yaml`),
whichever comes first. Prints progress every 10 trials. **Re-run this same cell
to continue** — it always resumes from whatever's already in `results/`, in this
session or a restored one.

Pilot target: ~240 LLM trials total (3 strategies x 2 axes x 40 reps) — expect
3-5 batches across sessions to finish the pilot.

In [ ]:
import os

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

from experiments.run_selection_trials import run_pilot

trial_csv, client_csv = run_pilot(tokenizer, model, device)


## Cell 5 — Push results out (last cell of every session, no exceptions)

Two steps:
1. Save the updated CSVs as this notebook's output, then create/update your
   `results/` Kaggle Dataset from that output (Kaggle: "Save Version" ->
   the CSVs under `/kaggle/working` become the new Dataset version — do this
   from the Kaggle UI after running this cell, or via the Kaggle API below).
2. Commit and push any code changes back to GitHub (only if you edited code
   this session — the pilot itself doesn't need to).

In [ ]:
import os, shutil

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

OUT_DIR = "/kaggle/working/phase4_results_out"
os.makedirs(OUT_DIR, exist_ok=True)
staged = []
for fname in ("phase4_pilot_trials.csv", "phase4_pilot_clients.csv"):
    src = os.path.join("results", fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, fname))
        staged.append(fname)
        print("staged for Dataset push:", fname)

if not staged:
    print("Nothing staged — results/ is empty. Did Cell 4 (run_pilot) actually run in this kernel?")

print("""
Now, from the Kaggle notebook UI:
  Save Version -> attach /kaggle/working/phase4_results_out as a new version
  of your results/ Kaggle Dataset (create it on the first session, update it
  on every session after).
""")


## Cell 6 — Power calculation (only once the pilot batch is done, ~240 trials)

Fits the observed phrasing-variant effect per axis and prints the trial count
Phase 5's Core matrix needs per (strategy, axis) cell for 80% power.

In [ ]:
!python analysis/phase4_power_calc.py results/phase4_pilot_clients.csv


---
# Phase 5 — Core Trial Matrix

Run this section only after Phase 4's pilot is fully done (240/240) and you've
looked at `analysis/phase4_power_calc_results.md`. This continues the SAME
results files by `trial_id` — Phase 4's 240 trials are already the first
40/cell of this matrix, nothing is re-run or duplicated.

Sized from the pilot: 120 trials/cell for formality, 230/cell for implied
geography (`configs/phase5_core_matrix.yaml`) — 1050 trials total. This is
the plan's largest time block; expect many bounded sessions. Per the plan,
Phase 5 needs zero GPU beyond the forward pass already used in Phase 4, so
this can run from a cheaper CPU session if you want to save GPU-hour quota
for other phases — re-run Cells 1-2 there too (the model still needs to be
loaded, just not on a GPU device).

**Do not run `analysis/phase4_power_calc.py` again as a stopping rule while
this matrix is still filling in** — the target trial counts were fixed once
from the pilot; re-checking significance after every batch and stopping
early would inflate the false-positive rate. Just run bounded batches until
the matrix is complete, then move to Phase 6's full analysis.

## Cell 7 — Run one bounded batch of the Core matrix

Same resume/flush/bounded-batch behavior as Cell 4, pointed at
`configs/phase5_core_matrix.yaml` instead. Re-run across sessions until
`run_pilot()` reports 1050/1050 total planned.

In [ ]:
import os

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

from experiments.run_selection_trials import run_pilot

trial_csv, client_csv = run_pilot(tokenizer, model, device, config_path="configs/phase5_core_matrix.yaml")


## Cell 8 — Push results out (same as Cell 5, run at the end of every Phase 5 session too)

In [ ]:
import os, shutil

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

OUT_DIR = "/kaggle/working/phase4_results_out"
os.makedirs(OUT_DIR, exist_ok=True)
staged = []
for fname in ("phase4_pilot_trials.csv", "phase4_pilot_clients.csv"):
    src = os.path.join("results", fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, fname))
        staged.append(fname)
        print("staged for Dataset push:", fname)

if not staged:
    print("Nothing staged — results/ is empty. Did Cell 7 (run_pilot) actually run in this kernel?")

print("""
Now, from the Kaggle notebook UI:
  Save Version -> attach /kaggle/working/phase4_results_out as a new version
  of your results/ Kaggle Dataset.
""")


---
# Phase 6 — Statistical Analysis

Run this once the Phase 5 Core matrix is fully collected (1050/1050) and
pushed out as a `results/` Dataset version. Fits, per (axis, selector)
cell — the 3 LLM strategies plus the numeric control, on identical pools —
a chi-square test and a cluster-robust logistic regression, then reads off
the paper's central contrast: an LLM strategy showing a phrasing effect
while the numeric control shows none, on the SAME pools.

In [ ]:
import os

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

!pip install -q matplotlib
!python analysis/phase6_analysis.py results/phase4_pilot_clients.csv


## Cell 10 — Display the results inline

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown

display(pd.read_csv("analysis/phase6_results.csv"))
display(Markdown(open("analysis/phase6_results.md").read()))
for axis in pd.read_csv("analysis/phase6_results.csv")["axis"].unique():
    display(Image(f"analysis/phase6_forest_{axis}.png"))


## Cell 11 — Package analysis outputs to download

Kaggle notebooks don't have push access to your GitHub by default, so this
just stages the outputs under Kaggle's output panel for you to download —
hand them (or the printed numbers) back so they can be committed.

In [ ]:
import shutil, os

REPO_DIR = "/kaggle/working/fairagent"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

OUT_DIR = "/kaggle/working/phase6_analysis_out"
os.makedirs(OUT_DIR, exist_ok=True)
for fname in os.listdir("analysis"):
    if fname.startswith("phase6_"):
        shutil.copy(os.path.join("analysis", fname), os.path.join(OUT_DIR, fname))
        print("staged for download:", fname)

print("\nDownload these from the Kaggle notebook's Output panel "
      "(/kaggle/working/phase6_analysis_out), then send them back.")
